# Bernacchi 2001: in vivo temperature responses of Rubisco kinetics

Reference: C. J. Bernacchi, E. L. Singsaas, C. Pimentel, A. R. Portis Jr, S. P. Long
"Improved temperature response functions for models of Rubisco-limited photosynthesis"
Plant, Cell & Environment 24 (2001): 253-259.

[doi: 10.1111/j.1365-3040.2001.00668.x](https://www.doi.org/10.1111/j.1365-3040.2001.00668.x)

The model is evaluated with the plain numpy implementation below, independent
of green-sloth's `model.ts`. Fig 2 compares other published sets of temperature
functions against data for sugar maple and is not reproduced.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

R = 0.008314  # kJ K^-1 mol^-1
# Table 1: c and dHa (kJ mol^-1) of Eqn 8
TABLE1 = {
    "Rd": (18.72, 46.39),
    "Vcmax": (26.35, 65.33),
    "Vomax": (22.98, 60.11),
    "Gamma_star": (19.02, 37.83),
    "Kc": (38.05, 79.43),
    "Ko": (20.30, 36.38),
}


def eqn8(name, Tk):
    c, dHa = TABLE1[name]
    return np.exp(c - dHa / (R * Tk))


def bernacchi(T_leaf, Ci=100.0, O=210.0, Vcmax25=44.1, Rd25=1.09) -> pd.DataFrame:
    T_leaf = np.asarray(T_leaf, dtype=float)
    Tk = T_leaf + 273.15
    Gamma_star, Kc, Ko = eqn8("Gamma_star", Tk), eqn8("Kc", Tk), eqn8("Ko", Tk)
    Vcmax = Vcmax25 * eqn8("Vcmax", Tk)
    Vomax = Vcmax25 * eqn8("Vomax", Tk)
    Rd = Rd25 * eqn8("Rd", Tk)
    vc = Vcmax * Ci / (Ci + Kc * (1 + O / Ko))  # Eqn 2
    vo = Vomax * O / (O + Ko * (1 + Ci / Kc))  # Eqn 3
    A = (1 - Gamma_star / Ci) * vc - Rd  # Eqn 4
    return pd.DataFrame(
        {"Gamma_star": Gamma_star, "Kc": Kc, "Ko": Ko, "Vcmax": Vcmax, "Vomax": Vomax,
         "Rd": Rd, "Vomax_Vcmax": Vomax / Vcmax, "vc": vc, "vo": vo, "A": A},
        index=pd.Index(T_leaf, name="T_leaf"),
    )

## Figure 1: temperature responses of the six parameters

Vc,max, Vo,max and Rd relative to their value at 25 °C (Vo,max relative to Vc,max).

In [ ]:
fig1 = bernacchi(np.linspace(10, 40, 301), Vcmax25=1.0, Rd25=1.0)
panels = [
    ("a", "Gamma_star", r"$\Gamma^*$ ($\mu$mol mol$^{-1}$)", (0, 100)),
    ("b", "Rd", r"$R_d$ ($\mu$mol m$^{-2}$ s$^{-1}$)", (0.5, 2.5)),
    ("c", ["Vcmax", "Vomax"], r"$V_{max}$ ($\mu$mol m$^{-2}$ s$^{-1}$)", (0, 5)),
    ("d", "Vomax_Vcmax", r"$V_{o,max}/V_{c,max}$", (0, 0.4)),
    ("e", "Kc", r"$K_c$ ($\mu$mol mol$^{-1}$)", (0, 2000)),
    ("f", "Ko", r"$K_o$ (mmol mol$^{-1}$)", (0, 2000)),
]
fig, axs = plt.subplots(3, 2, figsize=(7, 8), sharex=True)
for ax, (panel, cols, ylabel, ylim) in zip(axs.flat, panels):
    for col in [cols] if isinstance(cols, str) else cols:
        ax.plot(fig1.index, fig1[col], color="black")
    ax.set(ylabel=ylabel, ylim=ylim, xlim=(5, 45))
    ax.text(0.9, 0.08, f"{panel}.", transform=ax.transAxes)
for ax in axs[-1]:
    ax.set_xlabel("Temperature (°C)")
fig.tight_layout()
fig.savefig("figs/fig1.svg")
plt.show()

## Figure 3: Rubisco-limited assimilation of lemon

At Ci = 100 µmol mol⁻¹. The paper gives no Vc,max or Rd at 25 °C for its lemon
leaves; Vc,max,25 = 44.1 and Rd,25 = 1.09 µmol m⁻² s⁻¹ are fitted to its
modelled values (within 0.03 µmol m⁻² s⁻¹). The measured means are digitized
from the figure.

In [ ]:
LEMON_T = [10, 15, 20, 25, 30, 35, 40]
LEMON_MEASURED = [2.31, 2.68, 2.57, 2.23, 1.05, -0.37, -1.86]
fig3 = bernacchi(np.array(LEMON_T, dtype=float))
fig, ax = plt.subplots(figsize=(4.6, 4.0))
ax.plot(fig3.index, fig3["A"], "o-", color="black", label="Modelled values")
ax.plot(LEMON_T, LEMON_MEASURED, "s", color="black", markerfacecolor="none", label="Measured values")
ax.set(xlabel="Temperature (°C)", ylabel=r"$A$ ($\mu$mol m$^{-2}$ s$^{-1}$)", xlim=(5, 45), ylim=(-3, 3))
ax.legend(frameon=False, loc="lower left")
fig.tight_layout()
fig.savefig("figs/fig3.svg")
plt.show()